# Strands Agents with Bedrock AgentCore Code Interpreter

This lab demonstrates how to use Amazon Bedrock AgentCore Code Interpreter to give your AI agent the ability to execute Python code dynamically — applied to financial services use cases.

## Overview

In this lab, you will:
- Use the default Code Interpreter to run financial calculations in a sandbox
- Analyze transaction data for fraud patterns
- Calculate portfolio risk metrics (VaR, sector concentration)
- Create a custom Code Interpreter with network access for live market data

## Why Code Interpreter for FSI?

Financial services require:
- **Dynamic calculations** — Risk models, stress tests, scenario analysis
- **Data analysis** — Fraud detection, anomaly identification
- **Secure execution** — Sandboxed environment for sensitive financial data
- **Audit trail** — Every calculation is traceable

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore pandas

In [1]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Default Code Interpreter — Financial Calculations

The default Code Interpreter runs Python in a **sandboxed environment** with no network access. Perfect for secure financial calculations.

Let's test it with a portfolio risk calculation:

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

# Initialize the AgentCore Code Interpreter (default: sandboxed, no network)
agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create agent with default Code Interpreter (sandboxed)
risk_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="""You are a quantitative analyst assistant. You write and execute Python code 
    to perform financial calculations. Keep responses concise.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

risk_agent("Calculate the future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years.")

<thinking> To calculate the future value of an investment compounded monthly, we can use the formula:

Future Value = Principal * (1 + (rate / number of compounding periods per year)) ^ (number of years * number of compounding periods per year)

In this case, the principal is $2,000,000, the annual rate is 4.8%, the number of compounding periods per year is 12 (monthly), and the number of years is 5. 

I will use the code_interpreter tool to perform this calculation. </thinking>


Tool #1: code_interpreter
The future value of a $2,000,000 investment at a 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The future value of a $2,000,000 investment at a 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.'}], 'metadata': {'usage': {'inputTokens': 4160, 'outputTokens': 48, 'totalTokens': 4208}, 'metrics': {'latencyMs': 768, 'timeToFirstByteMs': 505}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'code_interpreter': ToolMetrics(tool={'toolUseId': 'tooluse_A9sUzHhiqOOBGjQ0N2vTG2', 'name': 'code_interpreter', 'input': {'code_interpreter_input': {'action': {'type': 'executeCode', 'code': 'principal = 2000000\nrate = 0.048\ncompounding_periods_per_year = 12\nyears = 5\nfuture_value = principal * (1 + (rate / compounding_periods_per_year)) ** (years * compounding_periods_per_year)\nprint(future_value)', 'language': 'python'}}}}, call_count=1, success_count=1, error_count=0, total_time=1.9234328269958496)}, cycle_durations=[4.2368292808532715, 0.8428599834442139], agent

## Part 2: Fraud Detection on Transaction Data

Now let's give the agent our synthetic transaction dataset and ask it to identify fraud patterns.

The dataset (`data/transactions.csv`) contains 25 transactions with several suspicious patterns:
- **Velocity attack** — Multiple high-value transactions within seconds
- **Geo-anomaly** — Transactions in different countries within minutes
- **Escalating amounts** — Progressively larger transactions (testing limits)
- **Unusual timing** — High-value transactions at 3am

In [4]:
# Load the transaction data so we can pass it to the agent
import pandas as pd

transactions_df = pd.read_csv("../data/transactions.csv")
print(f"Loaded {len(transactions_df)} transactions")
transactions_df.head()

Loaded 40 transactions


,transaction_id,timestamp,customer_id,amount,currency,merchant,category,location_city,location_country,card_type,is_online
0,TXN-001,2026-05-28 08:15:23,CUST-4421,12.5,AUD,Morning Brew Cafe,Food & Drink,Sydney,AU,debit,False
1,TXN-002,2026-05-28 08:17:45,CUST-4421,3200.0,AUD,TechWorld Electronics,Electronics,Lagos,NG,debit,True
2,TXN-003,2026-05-28 08:18:12,CUST-4421,2800.0,AUD,GiftCards Express,Gift Cards,Lagos,NG,debit,True
3,TXN-004,2026-05-28 08:19:01,CUST-4421,1500.0,AUD,Crypto Exchange XYZ,Financial Services,Moscow,RU,debit,True
4,TXN-005,2026-05-28 12:30:00,CUST-4421,15.8,AUD,Lunch Spot,Food & Drink,Sydney,AU,debit,False


In [7]:
# Pass the data as context and ask the agent to analyze it
transaction_data = transactions_df.to_csv(index=False)

risk_agent(f"""Write Python code to flag fraudulent transactions from this CSV data.

Rules (assign risk_score in parentheses):
- HIGH_AMOUNT (9): amount > 5000
- VELOCITY (6): same customer_id has 3+ rows within 5 minutes
- ODD_HOURS (4): hour between 0 and 5

Use io.StringIO to parse. Print a table: transaction_id, customer_id, amount, flag_reason, risk_score.
If a transaction matches multiple rules, combine them and sum the scores.
Sort by risk_score desc.

Data:
{transaction_data}""")


<thinking> To solve this problem, I will follow these steps:

1. Parse the CSV data using io.StringIO.
2. Iterate through each transaction and apply the rules to determine if it's fraudulent.
3. For each transaction, check if it matches any of the rules and assign the corresponding risk score.
4. If a transaction matches multiple rules, combine the reasons and sum the scores.
5. Store the results in a list of dictionaries.
6. Sort the list by risk_score in descending order.
7. Print the results in a table format.

I will use the code_interpreter tool to execute the Python code that performs these steps. </thinking> 
Tool #3: code_interpreter
Here are the flagged fraudulent transactions sorted by risk_score in descending order:

| transaction_id | customer_id | amount | flag_reason                  | risk_score |
|----------------|-------------|--------|------------------------------|------------|
| TXN-028        | CUST-6678   | 7500.00| HIGH_AMOUNT, ODD_HOURS, VELOCITY | 19         |


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here are the flagged fraudulent transactions sorted by risk_score in descending order:\n\n| transaction_id | customer_id | amount | flag_reason                  | risk_score |\n|----------------|-------------|--------|------------------------------|------------|\n| TXN-028        | CUST-6678   | 7500.00| HIGH_AMOUNT, ODD_HOURS, VELOCITY | 19         |\n| TXN-011        | CUST-9156   | 9999.99| HIGH_AMOUNT, VELOCITY        | 15         |\n| TXN-012        | CUST-9156   | 9999.99| HIGH_AMOUNT, VELOCITY        | 15         |\n| TXN-026        | CUST-6678   | 7500.00| HIGH_AMOUNT, ODD_HOURS       | 13         |\n| TXN-027        | CUST-6678   | 7500.00| HIGH_AMOUNT, ODD_HOURS       | 13         |\n| TXN-039        | CUST-3310   | 12000.00| HIGH_AMOUNT                  | 9          |\n| TXN-009        | CUST-9156   | 9999.99| HIGH_AMOUNT                  | 9          |\n| TXN-010        | CUST-9156   | 9

## Part 3: Portfolio Risk Analysis (VaR)

Let's analyze a portfolio using Value at Risk (VaR) — a standard risk metric in financial services.

We'll use the portfolio data from `data/portfolio.csv`.

In [8]:
portfolio_df = pd.read_csv("../data/portfolio.csv")
print(f"Loaded {len(portfolio_df)} positions")
portfolio_df.head(10)

Loaded 15 positions


,client_id,client_name,asset_class,ticker,units,purchase_price,current_price,currency,weight_pct,sector
0,CLI-001,Acme Super Fund,Equity,CBA.AX,15000,95.2,112.45,AUD,18.5,Financials
1,CLI-001,Acme Super Fund,Equity,BHP.AX,12000,42.8,45.60,AUD,12.2,Materials
2,CLI-001,Acme Super Fund,Equity,CSL.AX,3000,280.0,295.50,AUD,9.8,Healthcare
3,CLI-001,Acme Super Fund,Equity,WBC.AX,20000,22.5,25.80,AUD,8.6,Financials
4,CLI-001,Acme Super Fund,Equity,NAB.AX,18000,28.9,32.10,AUD,7.9,Financials
5,CLI-001,Acme Super Fund,Fixed Income,GOVT.AX,50000,100.0,98.50,AUD,15.0,Government Bonds
6,CLI-001,Acme Super Fund,Fixed Income,IAF.AX,30000,100.0,101.20,AUD,10.5,Corporate Bonds
7,CLI-001,Acme Super Fund,International,VGS.AX,8000,85.0,98.20,AUD,10.8,Global Equity
8,CLI-001,Acme Super Fund,Cash,CASH,500000,1.0,1.00,AUD,6.7,Cash
9,CLI-002,XYZ Treasury,Equity,AAPL,5000,175.0,198.50,USD,22.0,Technology


In [10]:
portfolio_data = portfolio_df.to_csv(index=False)

risk_agent(f"""Analyze this portfolio for risk metrics. Calculate:
1. Total portfolio value (current prices × units) for each client
2. Sector concentration — what % is in each sector? Flag if any sector > 30%
3. Unrealized P&L per position (current vs purchase price)
4. Asset class allocation (Equity vs Fixed Income vs Cash vs Other)

Present results as a clear summary with any risk warnings.

Portfolio data:
{portfolio_data}""")

Here is the analysis of the portfolio for risk metrics:

### Acme Super Fund (CLI-001)
1. **Total Portfolio Value**: AUD 5,011,100
   - CBA.AX: 15,000 units × 112.45 AUD = 1,686,750 AUD
   - BHP.AX: 12,000 units × 45.6 AUD = 547,200 AUD
   - CSL.AX: 3,000 units × 295.5 AUD = 886,500 AUD
   - WBC.AX: 20,000 units × 25.8 AUD = 516,000 AUD
   - NAB.AX: 18,000 units × 32.1 AUD = 577,800 AUD
   - GOVT.AX: 50,000 units × 98.5 AUD = 4,925,000 AUD
   - IAF.AX: 30,000 units × 101.2 AUD = 3,036,000 AUD
   - VGS.AX: 8,000 units × 98.2 AUD = 785,600 AUD
   - CASH: 500,000 units × 1.0 AUD = 500,000 AUD

2. **Sector Concentration**:
   - Financials: (18.5% + 8.6% + 7.9%) = 35% (Flagged for high concentration)
   - Materials: 12.2%
   - Healthcare: 9.8%
   - Government Bonds: 15.0%
   - Corporate Bonds: 10.5%
   - Global Equity: 10.8%
   - Cash: 6.7%

3. **Unrealized P&L per Position**:
   - CBA.AX: (112.45 - 95.2) × 15,000 = 258,750 AUD
   - BHP.AX: (45.6 - 42.8) × 12,000 = 33,600 AUD
   - CSL.AX: (

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here is the analysis of the portfolio for risk metrics:\n\n### Acme Super Fund (CLI-001)\n1. **Total Portfolio Value**: AUD 5,011,100\n   - CBA.AX: 15,000 units × 112.45 AUD = 1,686,750 AUD\n   - BHP.AX: 12,000 units × 45.6 AUD = 547,200 AUD\n   - CSL.AX: 3,000 units × 295.5 AUD = 886,500 AUD\n   - WBC.AX: 20,000 units × 25.8 AUD = 516,000 AUD\n   - NAB.AX: 18,000 units × 32.1 AUD = 577,800 AUD\n   - GOVT.AX: 50,000 units × 98.5 AUD = 4,925,000 AUD\n   - IAF.AX: 30,000 units × 101.2 AUD = 3,036,000 AUD\n   - VGS.AX: 8,000 units × 98.2 AUD = 785,600 AUD\n   - CASH: 500,000 units × 1.0 AUD = 500,000 AUD\n\n2. **Sector Concentration**:\n   - Financials: (18.5% + 8.6% + 7.9%) = 35% (Flagged for high concentration)\n   - Materials: 12.2%\n   - Healthcare: 9.8%\n   - Government Bonds: 15.0%\n   - Corporate Bonds: 10.5%\n   - Global Equity: 10.8%\n   - Cash: 6.7%\n\n3. **Unrealized P&L per Position**:\n   

## Part 4: Custom Code Interpreter with Network Access

The default Code Interpreter is sandboxed (no internet). For use cases that need live data (e.g., fetching real stock prices), we create a **custom Code Interpreter with network access**.

### Step 1: Initialize AgentCore Clients

In [12]:
from bedrock_agentcore._utils import endpoints
import boto3

region = boto3.session.Session().region_name

data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client('bedrock-agentcore-control',
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client('bedrock-agentcore',
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

print(f'✅ AgentCore clients initialized (region: {region})')

✅ AgentCore clients initialized (region: ap-southeast-2)


### Step 2: Create Custom Code Interpreter with Network Access

In [13]:
from botocore.exceptions import ClientError

interpreter_name = 'fsi_risk_analyzer'

try:
    interpreter_response = cp_client.create_code_interpreter(
        name=interpreter_name,
        description='FSI Code Interpreter with network access for live market data',
        networkConfiguration={'networkMode': 'PUBLIC'}
    )
    interpreter_id = interpreter_response['codeInterpreterId']
    print(f'✅ Created interpreter: {interpreter_id}')
except ClientError as e:
    if 'already exists' in str(e):
        for item in cp_client.list_code_interpreters()['codeInterpreterSummaries']:
            if item['name'] == interpreter_name:
                interpreter_id = item['codeInterpreterId']
                print(f'✅ Using existing interpreter: {interpreter_id}')
                break
    else:
        raise e

✅ Using existing interpreter: fsi_risk_analyzer-vORoi4eDnY


### Step 3: Create a Session and Test Live Data Access

In [14]:
# Create a session in the custom code interpreter
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id
)
session_id = session_response['sessionId']
print(f'✅ Session created: {session_id}')

# Helper to run code and extract output
def run_code(code, name='executeCode'):
    args = {'code': code, 'language': 'python'} if name == 'executeCode' else {'command': code}
    response = dp_client.invoke_code_interpreter(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id,
        name=name,
        arguments=args
    )
    for event in response.get('stream', []):
        if 'result' in event:
            content = event['result'].get('content', [])
            return '\n'.join(c.get('text', '') for c in content)
    return ''

# Install yfinance
print('Installing yfinance...')
run_code('pip install -q yfinance', name='executeCommand')

# Fetch live stock price
output = run_code("import yfinance as yf\ncba = yf.Ticker('CBA.AX')\ndata = cba.history(period='1d')\nprint(f'CBA.AX Live Price: ${data[\"Close\"].iloc[-1]:.2f} AUD')")
print(output)

✅ Session created: 01KTAKT1ZYDA1FYFNT10SRGEQ9
Installing yfinance...
CBA.AX Live Price: $162.88 AUD


### Step 4: Use Custom Code Interpreter with Strands Agent

In [16]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def execute_python(code: str) -> str:
    '''Execute Python code in a secure sandbox with internet access.

    Args:
        code: Python code to execute
    '''
    return run_code(code)

# Create agent with custom code interpreter
live_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a quantitative analyst with access to live market data.
    Execute Python code to fetch real-time prices using yfinance (already installed).
    Keep responses concise.''',
    tools=[execute_python],
)

live_agent('Fetch the current prices of CBA.AX and WBC.AX and compare their P/E ratios.')

<thinking> To fetch the current prices of CBA.AX and WBC.AX, I will use the yfinance library. After obtaining the prices, I will need to fetch their P/E ratios to compare them. However, yfinance does not directly provide P/E ratios, so I will need to fetch additional financial data to calculate the P/E ratios. </thinking>


Tool #1: execute_python
The current price of CBA.AX is 162.51 with a P/E ratio of 26.17, and the current price of WBC.AX is 34.9 with a P/E ratio of 17.19.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The current price of CBA.AX is 162.51 with a P/E ratio of 26.17, and the current price of WBC.AX is 34.9 with a P/E ratio of 17.19.'}], 'metadata': {'usage': {'inputTokens': 820, 'outputTokens': 58, 'totalTokens': 878}, 'metrics': {'latencyMs': 759, 'timeToFirstByteMs': 421}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'execute_python': ToolMetrics(tool={'toolUseId': 'tooluse_BfvmuSswCN9JzMC5e4erF2', 'name': 'execute_python', 'input': {'code': "import yfinance as yf\n\ncba = yf.Ticker('CBA.AX')\nwbc = yf.Ticker('WBC.AX')\n\ncba_info = cba.info\nwbc_info = wbc.info\n\ncba_price = cba_info['currentPrice']\nwbc_price = wbc_info['currentPrice']\n\ncba_pe = cba_info['trailingPE']\nwbc_pe = wbc_info['trailingPE']\n\nprint(f'CBA.AX Price: {cba_price}, P/E Ratio: {cba_pe}')\nprint(f'WBC.AX Price: {wbc_price}, P/E Ratio: {wbc_pe}')"}}, call_count=1, success_count=1, error_count=0, total_time=2.00

## Examining the Agent Loop

In [17]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {live_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta", max_width=60)
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan", max_width=40)
table.add_column("Tool Result", style="cyan", max_width=40)

for message in live_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], (text[-1][:200] + "...") if text and len(text[-1]) > 200 else (text[-1] if text else ""),
        tool_name[-1] if tool_name else "",
        (json.dumps(tool_input[-1])[:150] + "...") if tool_input else "",
        (json.dumps(tool_result[-1])[:150] + "...") if tool_result else ""
    )

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 2

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                      ┃ Tool Name      ┃ Tool Input                 ┃ Tool Result               ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ Fetch the current prices  │                │                            │                           │
│           │ of CBA.AX and WBC.AX and  │                │                            │                           │
│           │ compare their P/E ratios. │                │                            │                           │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ assistant │ <thinking> To fetch the   │ execute_python │ {"code": "import yfinance  │                           │
│           │ current prices of CBA.AX  │                │ as yf\n\ncba =             │                           │
│           │ and WBC.AX, I will use    │                │ yf.Ticker('CBA.AX')\nwbc = │                           │
│           │ the yfinance library.     │                │ yf.Ticker('WBC.AX')\n\ncb… │                           │
│           │ After obtaining the       │                │ = cba.info\nwbc_info =     │                           │
│           │ prices, I will need to    │                │ wbc.info\n\ncba_price =    │                           │
│           │ fetch their P/E ratios to │                │ cba...                     │                           │
│           │ compare them. However,    │                │                            │                           │
│           │ yfinance does ...         │                │                            │                           │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ user      │                           │                │                            │ {"text": "CBA.AX Price:   │
│           │                           │                │                            │ 162.51, P/E Ratio:        │
│           │                           │                │                            │ 26.16908\nWBC.AX Price:   │
│           │                           │                │                            │ 34.9, P/E Ratio:          │
│           │                           │                │                            │ 17.19212"}...             │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ assistant │ The current price of      │                │                            │                           │
│           │ CBA.AX is 162.51 with a   │                │                            │                           │
│           │ P/E ratio of 26.17, and   │                │                            │                           │
│           │ the current price of      │                │                            │                           │
│           │ WBC.AX is 34.9 with a P/E │                │                            │                           │
│           │ ratio of 17.19.           │                │                            │                           │
└───────────┴───────────────────────────┴────────────────┴────────────────────────────┴───────────────────────────┘

## Resource Cleanup (Optional)

Clean up the custom Code Interpreter to avoid charges:

In [ ]:
dp_client.stop_code_interpreter_session(
#     codeInterpreterIdentifier=interpreter_id,
#     sessionId=session_id
# )
cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
print('✅ Resources cleaned up')

## Summary

In this lab, you:

- ✅ Used the default Code Interpreter for secure financial calculations
- ✅ Analyzed transaction data for fraud patterns (velocity, geo-anomaly, timing)
- ✅ Calculated portfolio risk metrics (sector concentration, P&L, allocation)
- ✅ Created a custom Code Interpreter with network access for live market data
- ✅ Fetched real-time stock prices and compared bank P/E ratios

### FSI Takeaways

| Capability | FSI Application |
|-----------|----------------|
| Sandboxed execution | Secure risk calculations on sensitive data |
| Dynamic code generation | Ad-hoc analysis without pre-built reports |
| Network-enabled interpreter | Live market data, API integrations |
| Audit trail (agent loop) | Compliance — every calculation is traceable |

### Next: Lab 02 — Browser Automation
We'll use AgentCore Browser to monitor regulatory websites (APRA, ASX) and extract live financial data.